# Edge - Edge - Interactions 

Implementation of Edge - Edge interactions 

## Status 
1. Coinciding-Edge test case fails - ask Eugene to look along  

## To do 
1. Make a graph of a calling tree using GraphMakie.jl or similar; 
2. Make graphs of the test cases using Makie.jl;  
3. Revise coinciding edges; 

In [ ]:
#using Revise

In [ ]:
macro eval_bound_diff(func, domain)
    return quote
        local points = $(esc(domain))
        
        # Evaluate the function at both points
        local lb = $(esc(func))(first(points)...)
        local ub = $(esc(func))(last(points)...)
        
        # Calculate the difference between the two values
        ub - lb 
    end
end

foo(x) = x^2 

gnu(x,y) = x+y 
LB = (1.,20.); UB = (2.,20.)

@eval_bound_diff foo (1., 2.)
@eval_bound_diff gnu (LB, UB)

## Import Packages

In [ ]:
using StaticArrays 
using LinearAlgebra # provides the function norm 
using HCubature # provides adaptive numerical integration 

using Test # provides testing functionality 
using Plots

function hcubature_count(f, a, b; kws...)
    count = 0
    i = hcubature(a, b; kws...) do x
        count += 1
        # display(count)
        f(x)
    end
    return (i..., count)
end

const Point3D = SVector{3,Float64};

include("point-edge-interactions.jl"); # how to get rid of the semi-colom here?  
include("edge-edge-interactions.jl");

## Section 1: Introduction

Here we compute edge-edge interactions. More later. 

## Section 2: Computing Edge - Edge Interactions  

Here we compute the line (destination $\ell_1$) - line (source $\ell_2$) interaction 

$$ 
Q^{\phi_1,\theta_1}(\ell_1,\ell_2) = \int_{\ell_1} \int_{\ell_2} \frac{\phi_1(\mathbf{r}) \, \theta_1(\mathbf{r}')}
                                                         {\| \mathbf{r} - \mathbf{r}' \|}d\ell_1 \, d\ell_2' 
$$

This integral is singular in case that $\ell_1$ and $\ell_2$ intersect on the domain of integration. In this case, numerical integration might become cumbersome. We distinguish the following 4 cases 

### Case-1: Intersecting Edges (provide details on $I_{25}$, $I_{26}$, $I_{27}$ and $I_{28}$)

We have that 

$$
Q^{\phi_1,\theta_1}(\ell_1,\ell_2) = I_{25} - \varphi_1(\mathbf{s}) I_{26} - \theta_1(\mathbf{s}) I_{27} + \varphi_1(\mathbf{s}) \theta_1(\mathbf{s}) I_{28}
$$

where $I_{25}$, $I_{26}$, $I_{27}$ and $I_{28}$ are function of $(\ell_1,\ell_2)$.

See below. 

### Case-2: Crossing Edges (provide details on $I_{31}$, $I_{32}$, $I_{33}$ and $I_{34}$)

We have that

$$
Q^{\phi_1,\theta_1}(\ell_1,\ell_2) = I_{31} - \theta_1(\mathbf{s}) I_{32} - \hat{\varphi_1}(\mathbf{s}) I_{33} + \hat{\varphi_1}(\mathbf{s}) \theta_1(\mathbf{s}) I_{34}
$$

where $I_{31}$, $I_{32}$, $I_{33}$ and $I_{34}$ are function of $(\ell_1,\ell_2)$.

can be computed in terms of $I_8$ (see earlier), where $I_{31}$ is defined in terms of the one-dimensional integrals $I_{35}$, $I_{36}$, $I_{37}$ and $I_{38}$, where $I_{34}$ is a linear combination of four terms involving $I_3$ and where $I_{32}$ and $I_{33}$ are similar to each other; 

### Case-3: Coinciding/Extending Edges (provide details on $I_{41}$ and $I_{27}$)

We have that

$$
Q^{\phi_1,\theta_1}(\ell_1,\ell_2) = I_{41} - \theta_1(\mathbf{r}_2^1) I_{27} 
$$

where $I_{41}$ and $I_{27}$ are function of $(\ell_1,\ell_2)$.

### Case-4: Parallel Edges (provide details on $I_{47}$ and $I_{32}$)

We have that

$$
Q^{\phi_1,\theta_1}(\ell_1,\ell_2) = I_{47} - \theta_1(\hat{\mathbf{r}}_2^1) I_{32} 
$$

where $I_{47}$ and $I_{32}$ are function of $(\ell_1,\ell_2)$.

### How is the computation implemented?  

1. define EdgePair struct to hold two edges. This struct has three methods: distEdgePair to compute the distance between two edges,  dirEdgePair to compute the direction formed by the two edges and interEdgePair to compute the intersection of two edges;     
2. AbstractEdgePair (parent node in the type hierarchy): abstract type to dispatch on the concrete edgePair types outlined below. 
3. InterEdgePair, CrossEdgePair, ParalEdgePair and InlinEdgePair (child nodes). These types have methods that implement edge-edge interaction integrals. Composition is used to provide access to the edge data; 

### What Fails due to $R_0 = 0$  - Solved by truncating $R_0$ to small value 

The computation of edge-edge interactions for crossing edges fails in the computation of $I_{34}$. In this integral, terms with $R({\mathbf a}, \ell, h)$ fail due to $R_0$ (distance perpendicular to the edge) being zero.    

## Section 3: Test Case 

### Disclaimers 

This test assumes that
1. edges have length equal one: add more terms for non-unit length edges; 
2. edges are orthogonal: add more terms for non-orthogonal edges; 

### First Test for Case-1 Intersecting Edges: Shifted Unit Edge Along x-Axis and Unit Edge along y-Axis - Also works symbolically - Edit further to include results 

Assume (not that this test case is symmetrical wrt the coordinates $x$ and $y$)
* Edge-1 to extend along the x-Axis from the Point-1 ${\mathbf r}^1 = (0,0,0)$ to the Point-2 ${\mathbf r}^2 = (1,0,0)$. The destination position vector along Edge-1 can be written ${\mathbf r} = (x,0,0)$ for $0 \leq x \leq 1$. The direction vectors along Edge-1 are given by ${\mathbf \tau}_1 = {\mathbf e}_1 = (1,0,0)$;  
* Edge-2 to extend along the y-Axis from the Point-1 ${\mathbf r}^1 = (0,0,0)$ to the Point-2 ${\mathbf r}^2 = (0,1,0)$.  The source position vector along Edge-2 can be written as ${\mathbf r}_p = (0,y_p,0)$ for $0 \leq y_p \leq 1$. The direction vectors along Edge-2 are given by ${\mathbf \tau}_2 = {\mathbf e}_2 = (0,1,0)$; 
* the distance between Edge-1 (source) - Edge-2 (destination) can be written as $\|{\mathbf r} - {\mathbf r}_p \| = \sqrt{(x^2+y_p^2)}$;  

<b>Computation of $I_{25}$</b>

$$
I_{25} = \int_0^1 dx \int_0^1 dy_p \frac{x \, y_p}{\sqrt{(x^2+y_p^2)}} =  
$$

<b>Computation of $I_{26}$</b>

$$
I_{26} = \int_0^1 dx \int_0^1 dy_p \frac{y_p}{\sqrt{(x^2+y_p^2)}} =
$$

<b>Computation of $I_{27}$</b>

$$
I_{27} = \int_0^1 dx \int_0^1 dy_p \frac{x}{\sqrt{(x^2+y_p^2)}} =
$$

<b>Computation of $I_{28}$</b>

$$
I_{28} = \int_0^1 dx \int_0^1 dy_p \frac{1}{\sqrt{(x^2+y_p^2)}} =
$$

<b>Computation of $Q^{\phi_1,\theta_1}(\ell_1,\ell_2)$</b>  we have that 

$$
Q^{\varphi_1,\theta_1}(\ell_1,\ell_2) = \int_0^1 dx \int_0^1 dy_p \frac{\varphi_1({\mathbf r}) \, \theta_1({\mathbf r}_p)}{\|{\mathbf r} - {\mathbf r}_p \|} = \int_0^1 dx \int_0^1 dy_p \frac{(1-x) \, (1- y_p)}{\sqrt{(x^2+y_p^2)}} = I_{25} - \varphi_1({\mathbf s}) I_{26} - \theta_1({\mathbf s}) I_{27} + \varphi_1({\mathbf s}) \theta_1({\mathbf s}) I_{28} = I_{25} - I_{26} - I_{27} + I_{28} = ... \, . 
$$


In [ ]:
# return first edge 
r11 = Point3D(0.,0.,0.,); r12 = Point3D(0.,0.,1.); 
e1hat = Edge3D(r1 = r11,r2 = r12)

# return second edge 
r21 = Point3D(0.,0.,0.,); r22 = Point3D(0.,1.,0.); 
e2 = Edge3D(r1 = r21,r2 = r22)

# return point of intersection 
s = Point3D(0.,0.,0.,);

In [ ]:
I25integrand(lmbd) = (lmbd[1]*lmbd[2]) / sqrt(lmbd[1]^2 + lmbd[2]^2) 
I25testvalnum = hcubature(lmbd -> I25integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
svec = [0,0,0]
println(" num. I25 = ", I25testvalnum)
#println(" ana. I25 = ", I25testvalana)
#@test I25testvalnum ≈ I25testvalana

I26integrand(lmbd) = lmbd[2] / sqrt(lmbd[1]^2 + lmbd[2]^2) 
I26testvalnum = hcubature(lmbd -> I26integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
svec = [0,0,0]
println(" num. I26 = ", I26testvalnum)
#println(" ana. I26 = ", I26testvalana)
#@test I26testvalnum ≈ I26testvalana

I27integrand(lmbd) = lmbd[1] / sqrt(lmbd[1]^2 + lmbd[2]^2) 
I27testvalnum = hcubature(lmbd -> I27integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
svec = [0,0,0]
println(" num. I27 = ", I27testvalnum)
#println(" ana. I27 = ", I27testvalana)
#@test I27testvalnum ≈ I27testvalana

I28integrand(lmbd) = 1 / sqrt(lmbd[1]^2 + lmbd[2]^2) 
I28testvalnum = hcubature(lmbd -> I28integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
svec = [0,0,0]
println(" num. I28 = ", I28testvalnum)
#println(" ana. I28 = ", I28testvalana)
#@test I28testvalnum ≈ I28testvalana

Qintegrand(lmbd) = (1-lmbd[1])*(1-lmbd[2])/ sqrt(lmbd[1]^2 + lmbd[2]^2) 
Qtestvalnum = hcubature(lmbd -> Qintegrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
#Qtestvalana = I25testvalana - I26testvalana - I27testvalana + I28testvalana
Qtestvalana = I25testvalnum - I26testvalnum - I27testvalnum + I28testvalnum 
println(" num. Qnum = ", Qtestvalnum)
println(" ana. Qana = ", Qtestvalana)
@test Qtestvalnum ≈ Qtestvalana


### First Test for Case-2 Crossing Edges: Shifted Unit Edge Along x-Axis and Shifted Unit Edge along y-Axis - Also works symbolically - Edit further to include results

Assume (not that this test case is symmetrical wrt the coordinates $x$ and $y$)
* Edge-1 to extend along the x-Axis from the Point-1 ${\mathbf r}^1 = (1,0,0)$ to the Point-2 ${\mathbf r}^2 = (2,0,0)$. The destination position vector along Edge-1 can be written ${\mathbf r} = (x,0,0)$ for $1 \leq x \leq 2$. The direction vectors along Edge-1 are given by ${\mathbf \tau}_1 = {\mathbf e}_1 = (1,0,0)$;  
* Edge-2 to extend along the y-Axis from the Point-1 ${\mathbf r}^1 = (0,1,1)$ to the Point-2 ${\mathbf r}^2 = (0,2,1)$.  The source position vector along Edge-2 can be written as ${\mathbf r}_p = (0,y_p,1)$ for $1 \leq y_p \leq 2$. The direction vectors along Edge-2 are given by ${\mathbf \tau}_2 = {\mathbf e}_2 = (0,1,0)$; 
* the distance between Edge-1 (source) - Edge-2 (destination) can be written as $\|{\mathbf r} - {\mathbf r}_p \| = \sqrt{(x^2+y_p^2+1)}$.  
* the height is $h=1$. The point of intersection of the projected Edge-1 and Edge-2 is given by ${\mathbf s} = (0,0,1)$; 

<b>Computation of $I_{35}$ $R_0 = 2$</b>

$$
I_{35} = \int_1^2 y \, f_3(1, \sqrt{4+y^2}) \, dy 
= \left[ I_{13}(s,1,1) \right]_{s=1}^{s=2}
$$

<b>Computation of $I_{36}$ $R_0 = 1$</b>

$$
I_{36} = \int_1^2 y \, f_3(1, \sqrt{1+y^2}) \, dy
= \left[ I_{13}(s,1,0) \right]_{s=1}^{s=2}
$$

<b>Computation of $I_{37}$  $R_0 = 2$</b>

$$
I_{37} = \int_1^2 x \, f_3(1, \sqrt{4+x^2}) \, dx
= \left[ I_{13}(s,1,2) \right]_{s=1}^{s=2}
$$

<b>Computation of $I_{38}$   $R_0 = 1$</b>

$$
I_{38} = \int_1^2 x \, f_3(1, \sqrt{1+x^2}) \, dy
= \left[ I_{13}(s,1,1) \right]_{s=1}^{s=2}
$$

<b>Computation of $I_{31}$</b>  we have that 
* on the projected edge $\hat{\ell}_1$, ${\mathbf r} = (x,0,1)$ for $1 \leq x \leq 2$, thus ${\mathbf r} - {\mathbf s} = (x,0,0)$, and therefore $({\mathbf r} - {\mathbf s}) \cdot {\mathbf e}_1 = x$;
* on $\ell_2$, ${\mathbf r}_p = (0,y_p,1)$ for $1 \leq y_p \leq 2$, thus ${\mathbf r}_p - {\mathbf s} = (0,y_p,0)$, and therefore $({\mathbf r}_p - {\mathbf s}) \cdot {\mathbf e}_2 = y_p$
* the distance is given by $h^2 + \|{\mathbf r} - {\mathbf r}_p \|^2 = h^2+x^2+y_p^2 = x^2+y_p^2+1$

$$
I_{31} = \int_1^2 dx \int_1^2 dy_p \frac{x \, y_p}{\sqrt{x^2+y_p^2+1}}
$$

* in comparing with the analytical expression, we explicitly rely on $\| \hat{\ell}_1 \| = \| \ell_2 \| = 1 $; 

<b>Computation of $I_{32}$</b>  we have that 

$$
I_{32} = \int_1^2 dx \int_1^2 dy_p \frac{x}{\sqrt{x^2+y_p^2+1}}
$$

<b>Computation of $I_{33}$</b>  we have that 

$$
I_{33} = \int_1^2 dx \int_1^2 dy_p \frac{y_p}{\sqrt{x^2+y_p^2+1}}
$$

<b>Computation of $I_{34}$</b>  we have that 

$$
I_{34} = \int_1^2 dx \int_1^2 dy_p \frac{1}{\sqrt{x^2+y_p^2+1}}
$$

<b>Computation of $Q^{\phi_1,\theta_1}(\ell_1,\ell_2)$</b>  we have that $\theta_1({\mathbf s}) = 1$ and $\hat{\varphi}_1({\mathbf s}) = 1$ (not sure why these values are the correct ones) 

$$
Q^{\phi_1,\theta_1}(\ell_1,\ell_2) = \int_1^2 dx \int_1^2 dy_p \frac{\varphi_1({\mathbf r}) \, \theta_1({\mathbf r}_p)}{\|{\mathbf r} - {\mathbf r}_p \|} = \int_1^2 dx \int_1^2 dy_p \frac{x \, y_p}{\sqrt{(x^2+y_p^2+2)}} = I_{31} - \theta_1({\mathbf s}) I_{32} - \hat{\varphi}_1({\mathbf s}) I_{33} + \hat{\varphi}_1({\mathbf s}) \theta_1({\mathbf s}) I_{34} = I_{31} - I_{32} - I_{33} + I_{34} = 0.0978 ... \, . 
$$

In [ ]:
# return first edge 
r11 = Point3D(1.,0.,1.,); r12 = Point3D(2.,0.,1.); 
e1hat = Edge3D(r1 = r11,r2 = r12)

# return second edge 
r21 = Point3D(0.,1.,1.,); r22 = Point3D(0.,2.,1.); 
e2 = Edge3D(r1 = r21,r2 = r22)

h = 1; 
[R(e1hat.r1,e2,h), R(e1hat.r2,e2,h),R(e2.r1,e1hat,h), R(e2.r2,e1hat,h)]   

In [ ]:
I35integrand(y) = y[1]*f3(1,sqrt(4+y[1]^2))
I35testvalnum = hcubature(y -> I35integrand(y), (1,), (2,);rtol=1e-10,atol=1e-10)[1]
smin = 1.; smax = 2.; h = 1.; R0 = 2.  
LB = (smin,h,R0); UB = (smax,h,R0)
I35testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I35 = ", I35testvalnum)
println(" ana. I35 = ", I35testvalana)
@test I35testvalnum ≈ I35testvalana

I36integrand(y) = y[1]*f3(1,sqrt(1+y[1]^2))
I36testvalnum = hcubature(y -> I36integrand(y), (1,), (2,);rtol=1e-10,atol=1e-10)[1]
smin = 1.; smax = 2.; h = 1.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
I36testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I36 = ", I36testvalnum)
println(" ana. I36 = ", I36testvalana)
@test I36testvalnum ≈ I36testvalana

I37integrand(x) = x[1]*f3(1,sqrt(4+x[1]^2))
I37testvalnum = hcubature(x -> I37integrand(x), (1,), (2,);rtol=1e-10,atol=1e-10)[1]
smin = 1.; smax = 2.; h = 1.; R0 = 2.  
LB = (smin,h,R0); UB = (smax,h,R0)
I37testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I37 = ", I37testvalnum)
println(" ana. I37 = ", I37testvalana)
@test I37testvalnum ≈ I37testvalana

I38integrand(x) = x[1]*f3(1,sqrt(1+x[1]^2))
I38testvalnum = hcubature(x -> I38integrand(x), (1,), (2,);rtol=1e-10,atol=1e-10)[1]
smin = 1.; smax = 2.; h = 1.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
I38testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I38 = ", I38testvalnum)
println(" ana. I38 = ", I38testvalana)
@test I38testvalnum ≈ I38testvalana

I31integrand(lmbd) = lmbd[1]*lmbd[2]/ sqrt(lmbd[1]^2 + lmbd[2]^2 + 1) 
I31testvalnum = hcubature(lmbd -> I31integrand(lmbd), (1,1), (2,2);rtol=1e-10,atol=1e-10)[1]
svec = [0,0,1] 
I31testvalana = norm(e1hat.r2-svec)^2*I35testvalana 
I31testvalana += -norm(e1hat.r1-svec)^2*I36testvalana  
I31testvalana += norm(e2.r2-svec)^2*I37testvalana
I31testvalana += -norm(e2.r1-svec)^2*I38testvalana
println(" num. I31 = ", I31testvalnum)
println(" ana. I31 = ", I31testvalana)
@test I31testvalnum ≈ I31testvalana

I32integrand(lmbd) = lmbd[1] / sqrt(lmbd[1]^2 + lmbd[2]^2 + 1) 
I32testvalnum = hcubature(lmbd -> I32integrand(lmbd), (1,1), (2,2);rtol=1e-10,atol=1e-10)[1]
h = 1.; svec = [0,0,1]
smin = 1.; smax = 2.; R0 = 2.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term1 = norm(e1hat.r2-svec)^2*factor 
println("   I32ana term-1 = ", term1)
smin = 1.; smax = 2.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term2 = -norm(e1hat.r1-svec)^2*factor 
println("   I32ana term-2 = ", term2)
smin = 1.; smax = 2.; R0 = 2.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e2.r2-svec,e1hat.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term3 = dot(e2.r2-svec,e2.tau)*factor 
println("   I32ana term-3 = ", term3)
smin = 1.; smax = 2.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e2.r1-svec,e1hat.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term4 = -dot(e2.r1-svec,e2.tau)*factor
println("   I32ana term-4 = ", term4)
I32testvalana = term1 + term2 + term3 + term4  
println(" num. I32 = ", I32testvalnum)
println(" ana. I32 = ", I32testvalana)
@test I32testvalnum ≈ I32testvalana

I33integrand(lmbd) = lmbd[2] / sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
I33testvalnum = hcubature(lmbd -> I33integrand(lmbd), (1,1), (2,2);rtol=1e-10,atol=1e-10)[1]
h = 1.; svec = [0,0,1]
# upper bound for Edge-2
smin = 1.; smax = 2.; R0 = 2.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term1 = norm(e2.r2-svec)^2*factor 
println("   I33ana term-1 = ", term1)
# lower bound for Edge-2
smin = 1.; smax = 2.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term2 = -norm(e2.r1-svec)^2*factor
println("   I33ana term-1 = ", term2)
# upper bound for Edge-1
smin = 1.; smax = 2.; R0 = 2.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e1hat.r2-svec,e2.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term3 = dot(e1hat.r2-svec,e1hat.tau)*factor 
println("   I33ana term-3 = ", term3)
# lower bound for Edge-1
smin = 1.; smax = 2.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e1hat.r1-svec,e2.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term4 = -dot(e1hat.r1-svec,e1hat.tau)*factor
println("   I33ana term-4 = ", term4)
I33testvalana = term1 + term2 + term3 + term4 
println(" num. I33 = ", I33testvalnum)
println(" ana. I33 = ", I33testvalana)
#@test I33testvalnum ≈ I33testvalana

I34integrand(lmbd) = 1 / sqrt(lmbd[1]^2 + lmbd[2]^2 + 1) 
I34testvalnum = hcubature(lmbd -> I34integrand(lmbd), (1,1), (2,2);rtol=1e-10,atol=1e-10)[1]
h = 1.; svec = [0,0,1]
I34testvalana = dot(e1hat.r2-svec,e1hat.tau)*R(e1hat.r2,e2,h)
I34testvalana += -dot(e1hat.r1-svec,e1hat.tau)*R(e1hat.r1,e2,h)
I34testvalana += dot(e2.r2-svec,e2.tau)*R(e2.r2,e1hat,h)
I34testvalana += -dot(e2.r1-svec,e2.tau)*R(e2.r1,e1hat,h)
println(" num. I34 = ", I34testvalnum)
println(" ana. I34 = ", I34testvalana)
@test I34testvalnum ≈ I34testvalana

Qintegrand(lmbd) = (1-lmbd[1])*(1-lmbd[2])/ sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
Qtestvalnum = hcubature(lmbd -> Qintegrand(lmbd), (1,1), (2,2);rtol=1e-10,atol=1e-10)[1]
Qtestvalana = I31testvalana - I32testvalana - I33testvalana + I34testvalana
println(" num. Qnum = ", Qtestvalnum)
println(" ana. Qana = ", Qtestvalana)
@test Qtestvalnum ≈ Qtestvalana

### Second Test for Case-2 Crossing Edges: Unit Edge Along x-Axis and Shifted Unit Edge along y-Axis

Assume

* Edge-1 to extend along the x-Axis from the Point-1 ${\mathbf r}^1 = (0,0,0)$ to the Point-2 ${\mathbf r}^2 = (1,0,0)$. The destination position vector along Edge-1 can be written ${\mathbf r} = (x,0,0)$ for $0 \leq x \leq 1$. The direction vectors along Edge-1 are given by ${\mathbf \tau}_1 = {\mathbf e}_1 = (1,0,0)$;  
* Edge-2 to extend along the y-Axis from the Point-1 ${\mathbf r}^1 = (0,1,1)$ to the Point-2 ${\mathbf r}^2 = (0,2,1)$.  The source position vector along Edge-2 can be written as ${\mathbf r}_p = (0,y_p,1)$ for $1 \leq y_p \leq 2$. The direction vectors along Edge-2 are given by ${\mathbf \tau}_2 = {\mathbf e}_2 = (0,1,0)$; 
* the distance between Edge-1 (source) - Edge-2 (destination) can be written as $\|{\mathbf r} - {\mathbf r}_p \| = \sqrt{(x^2+y_p^2+1)}$.  
* the height is $h=1$. The point of intersection of the projected Edge-1 and Edge-2 is given by ${\mathbf s} = (0,0,1)$; 

<b>Computation of $I_{35}$ $R_0 = 1$</b>

$$
I_{35} = \int_1^2 y \, f_3(1, \sqrt{1+y^2}) \, dy 
= \left[ I_{13}(s,1,1) \right]_{s=1}^{s=2}
$$

<b>Computation of $I_{36}$ $R_0 = 0$</b>

$$
I_{36} = \int_1^2 y \, f_3(1, \sqrt{y^2}) \, dy
= \left[ I_{13}(s,1,0) \right]_{s=1}^{s=2}
$$

<b>Computation of $I_{37}$  $R_0 = 2$</b>

$$
I_{37} = \int_0^1 x \, f_3(1, \sqrt{4+x^2}) \, dx
= \left[ I_{13}(s,1,2) \right]_{s=0}^{s=1}
$$

<b>Computation of $I_{38}$   $R_0 = 1$</b>

$$
I_{38} = \int_0^1 x \, f_3(1, \sqrt{1+x^2}) \, dy
= \left[ I_{13}(s,1,1) \right]_{s=0}^{s=1}
$$

<b>Computation of $I_{31}$</b>  we have that 
* on the projected edge $\hat{\ell}_1$, ${\mathbf r} = (x,0,1)$ for $0 \leq x \leq 1$, thus ${\mathbf r} - {\mathbf s} = (x,0,0)$, and therefore $({\mathbf r} - {\mathbf s}) \cdot {\mathbf e}_1 = x$;
* on $\ell_2$, ${\mathbf r}_p = (0,y_p,1)$ for $1 \leq y_p \leq 2$, thus ${\mathbf r}_p - {\mathbf s} = (0,y_p,0)$, and therefore $({\mathbf r}_p - {\mathbf s}) \cdot {\mathbf e}_2 = y_p$
* the distance is given by $h^2 + \|{\mathbf r} - {\mathbf r}_p \|^2 = x^2+y_p^2+1$

$$
I_{31} = \int_0^1 dx \int_1^2 dy_p \frac{x \, y_p}{\sqrt{x^2+y_p^2+1}}
$$

<b>Computation of $I_{32}$</b> we have that 

$$
I_{32} = \int_0^1 dx \int_1^2 dy_p \frac{x}{\sqrt{x^2+y_p^2+1}}
$$

<b>Computation of $I_{33}$</b> we have that

$$
I_{33} = \int_0^1 dx \int_1^2 dy_p \frac{y_p}{\sqrt{x^2+y_p^2+1}}
$$

<b>Computation of $I_{34}$</b> we have that

$$
I_{34} = \int_0^1 dx \int_1^2 dy_p \frac{1}{\sqrt{x^2+y_p^2+1}}
$$

<b>Computation of $Q^{\phi_1,\theta_1}(\ell_1,\ell_2)$</b> we have that 

$$
Q^{\phi_1,\theta_1}(\ell_1,\ell_2) = \int_0^1 dx \int_1^2 dy_p \frac{\varphi_1({\mathbf r}) \, \theta_1({\mathbf r}_p)}{\|{\mathbf r} - {\mathbf r}_p \|} = \int_0^1 dx \int_1^2 dy_p \frac{x \, y_p}{\sqrt{(x^2+y_p^2+1)}} = I_{31} - I_{32} - I_{33} + I_{34} = - 0.127 ...\, . 
$$

In [ ]:
# return first edge 
r11 = Point3D(0.,0.,1.,); r12 = Point3D(1.,0.,1.); 
e1hat = Edge3D(r1 = r11,r2 = r12)

# return second edge 
r21 = Point3D(0.,1.,1.,); r22 = Point3D(0.,2.,1.); 
e2 = Edge3D(r1 = r21,r2 = r22)

h = 1; 
[R(e1hat.r1,e2,h), R(e1hat.r2,e2,h),R(e2.r1,e1hat,h), R(e2.r2,e1hat,h)]   

In [ ]:
I35integrand(y) = y[1]*f3(1,sqrt(1+y[1]^2))
I35testvalnum = hcubature(y -> I35integrand(y), (1,), (2,);rtol=1e-10,atol=1e-10)[1]
smin = 1.; smax = 2.; h = 1.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
I35testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I35 = ", I35testvalnum)
println(" ana. I35 = ", I35testvalana)
@test I35testvalnum ≈ I35testvalana

I36integrand(y) = y[1]*f3(1,sqrt(y[1]^2))
I36testvalnum = hcubature(y -> I36integrand(y), (1,), (2,);rtol=1e-10,atol=1e-10)[1]
smin = 1.; smax = 2.; h = 1.; R0 = 0.  
LB = (smin,h,R0); UB = (smax,h,R0)
I36testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I36 = ", I36testvalnum)
println(" ana. I36 = ", I36testvalana)
@test I36testvalnum ≈ I36testvalana

I37integrand(x) = x[1]*f3(1,sqrt(4+x[1]^2))
I37testvalnum = hcubature(x -> I37integrand(x), (0,), (1,);rtol=1e-10,atol=1e-10)[1]
smin = 0.; smax = 1.; h = 1.; R0 = 2.  
LB = (smin,h,R0); UB = (smax,h,R0)
I37testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I37 = ", I37testvalnum)
println(" ana. I37 = ", I37testvalana)
@test I37testvalnum ≈ I37testvalana

I38integrand(x) = x[1]*f3(1,sqrt(1+x[1]^2))
I38testvalnum = hcubature(x -> I38integrand(x), (0,), (1,);rtol=1e-10,atol=1e-10)[1]
smin = 0.; smax = 1.; h = 1.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
I38testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I38 = ", I38testvalnum)
println(" ana. I38 = ", I38testvalana)
@test I38testvalnum ≈ I38testvalana

I31integrand(lmbd) = lmbd[1]*lmbd[2]/ sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
I31testvalnum = hcubature(lmbd -> I31integrand(lmbd), (0,1), (1,2);rtol=1e-10,atol=1e-10)[1]
svec = [0,0,1] 
I31testvalana = norm(e1hat.r2-svec)^2*I35testvalana 
I31testvalana += -norm(e1hat.r1-svec)^2*I36testvalana  
I31testvalana += norm(e2.r2-svec)^2*I37testvalana
I31testvalana += -norm(e2.r1-svec)^2*I38testvalana
println(" num. I31 = ", I31testvalnum)
println(" ana. I31 = ", I31testvalana)
@test I31testvalnum ≈ I31testvalana

I32integrand(lmbd) = lmbd[1]/ sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
I32testvalnum = hcubature(lmbd -> I32integrand(lmbd), (0,1), (1,2);rtol=1e-10,atol=1e-10)[1]
h = 1.; svec = [0,0,1]
# upper bound for Edge-2
smin = 1.; smax = 2.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term1 = norm(e1hat.r2-svec)^2*factor
# lower bound for Edge-2
smin = 1.; smax = 2.; R0 = 0.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term2 = -norm(e1hat.r1-svec)^2*factor 
# upper bound for Edge-1
smin = 0.; smax = 1.; R0 = 2.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e2.r2-svec,e1hat.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term3 = dot(e2.r2-svec,e2.tau)*factor 
# lower bound for Edge-1
smin = 0.; smax = 1.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e2.r1-svec,e1hat.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term4 = -dot(e2.r1-svec,e2.tau)*factor
I32testvalana = term1 + term2 + term3 + term4  
println(" num. I32 = ", I32testvalnum)
println(" ana. I32 = ", I32testvalana)
@test I32testvalnum ≈ I32testvalana

I33integrand(lmbd) = lmbd[2]/ sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
I33testvalnum = hcubature(lmbd -> I33integrand(lmbd), (0,1), (1,2);rtol=1e-10,atol=1e-10)[1]
h = 1.; svec = [0,0,1]
# upper bound for Edge-1
smin = 0.; smax = 1.; R0 = 2.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term1 = norm(e2.r2-svec)^2*factor 
# lower bound for Edge-1
smin = 0.; smax = 1.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term2 = -norm(e2.r1-svec)^2*factor 
# upper bound for Edge-2
smin = 1.; smax = 2.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e1hat.r2-svec,e2.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term3 = dot(e1hat.r2-svec,e1hat.tau)*factor 
# lower bound for Edge-2
smin = 1.; smax = 2.; R0 = 0.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e1hat.r1-svec,e2.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term4 = -dot(e1hat.r1-svec,e1hat.tau)*factor
I33testvalana = term1 + term2 + term3 + term4 
println(" num. I33 = ", I33testvalnum)
println(" ana. I33 = ", I33testvalana)
@test I33testvalnum ≈ I33testvalana

I34integrand(lmbd) = 1 / sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
I34testvalnum = hcubature(lmbd -> I34integrand(lmbd), (0,1), (1,2);rtol=1e-10,atol=1e-10)[1]
h = 1.; svec = [0,0,1]
I34testvalana = dot(e1hat.r2-svec,e1hat.tau)*R(e1hat.r2,e2,h)
I34testvalana += -dot(e1hat.r1-svec,e1hat.tau)*R(e1hat.r1,e2,h)
I34testvalana += dot(e2.r2-svec,e2.tau)*R(e2.r2,e1hat,h)
I34testvalana += -dot(e2.r1-svec,e2.tau)*R(e2.r1,e1hat,h)
println(" num. I34 = ", I34testvalnum)
println(" ana. I34 = ", I34testvalana)
@test I34testvalnum ≈ I34testvalana

Qtestintegrand(lmbd) = (1-lmbd[1])*(1-lmbd[2])/ sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
Qtestvalnum = hcubature(lmbd -> Qtestintegrand(lmbd), (0,1), (1,2);rtol=1e-10,atol=1e-10)[1]
Qtestvalana = I31testvalana - I32testvalana - I33testvalana + I34testvalana
println(" num. Qnum = ", Qtestvalnum)
println(" ana. Qana = ", Qtestvalana)
@test Qtestvalnum ≈ Qtestvalana


### Third Test for Case-2 Crossing Edges: Unit Edge Along x-Axis and Shifted Unit Edge along y-Axis

Assume (this test case is again symmetrical wrt the cooordinates $x$ and $y$)

* Edge-1 to extend along the x-Axis from the Point-1 ${\mathbf r}^1 = (0,0,0)$ to the Point-2 ${\mathbf r}^2 = (1,0,0)$. The destination position vector along Edge-1 can be written ${\mathbf r} = (x,0,0)$ for $0 \leq x \leq 1$. The direction vectors along Edge-1 are given by ${\mathbf \tau}_1 = {\mathbf e}_1 = (1,0,0)$;  
* Edge-2 to extend along the y-Axis from the Point-1 ${\mathbf r}^1 = (0,0,1)$ to the Point-2 ${\mathbf r}^2 = (0,0,1)$.  The source position vector along Edge-2 can be written as ${\mathbf r}_p = (0,y_p,1)$ for $0 \leq y_p \leq 1$. The direction vectors along Edge-2 are given by ${\mathbf \tau}_2 = {\mathbf e}_2 = (0,1,0)$; 
* the distance between Edge-1 (source) - Edge-2 (destination) can be written as $\|{\mathbf r} - {\mathbf r}_p \| = \sqrt{(x^2+y_p^2+1)}$.  
* the height is $h=1$. The point of intersection of the projected Edge-1 and Edge-2 is given by ${\mathbf s} = (0,0,1)$; 

<b>Computation of $I_{35}$ $R_0 = 1$</b>

$$
I_{35} = \int_0^1 y \, f_3(1, \sqrt{1+y^2}) \, dy 
= \left[ I_{13}(s,1,1) \right]_{s=0}^{s=1}
$$

<b>Computation of $I_{36}$ $R_0 = 0$</b>

$$
I_{36} = \int_0^1 y \, f_3(1, \sqrt{y^2}) \, dy
= \left[ I_{13}(s,1,0) \right]_{s=0}^{s=1}
$$

<b>Computation of $I_{37}$  $R_0 = 1$</b>

$$
I_{37} = \int_0^1 x \, f_3(1, \sqrt{1+x^2}) \, dx
= \left[ I_{13}(s,1,1) \right]_{s=0}^{s=1}
$$

<b>Computation of $I_{38}$   $R_0 = 0$</b>

$$
I_{38} = \int_0^1 x \, f_3(1, \sqrt{x^2}) \, dy
= \left[ I_{13}(s,1,0) \right]_{s=0}^{s=1}
$$

<b>Computation of $I_{31}$</b>  we have that 
* on the projected edge $\hat{\ell}_1$, ${\mathbf r} = (x,0,1)$ for $0 \leq x \leq 1$, thus ${\mathbf r} - {\mathbf s} = (x,0,0)$, and therefore $({\mathbf r} - {\mathbf s}) \cdot {\mathbf e}_1 = x$;
* on $\ell_2$, ${\mathbf r}_p = (0,y_p,1)$ for $1 \leq y_p \leq 2$, thus ${\mathbf r}_p - {\mathbf s} = (0,y_p,0)$, and therefore $({\mathbf r}_p - {\mathbf s}) \cdot {\mathbf e}_2 = y_p$
* the distance is given by $h^2 + \|{\mathbf r} - {\mathbf r}_p \|^2 = x^2+y_p^2+2$

$$
I_{31} = \int_0^1 dx \int_0^1 dy_p \frac{x \, y_p}{\sqrt{x^2+y_p^2+2}}
$$

<b>Computation of $I_{32}$</b>

<b>Computation of $I_{33}$</b>

<b>Computation of $I_{34}$</b>

<b>Computation of $Q^{\phi_1,\theta_1}(\ell_1,\ell_2)$</b>

$$
Q^{\phi_1,\theta_1}(\ell_1,\ell_2) = \int_0^1 dx \int_0^1 dy_p \frac{\varphi_1({\mathbf r}) \, \theta_1({\mathbf r}_p)}{\|{\mathbf r} - {\mathbf r}_p \|} = \int_0^1 dx \int_0^1 dy_p \frac{x \, y_p}{\sqrt{(x^2+y_p^2+1)}} \, . 
$$

In [ ]:
# return first edge 
r11 = Point3D(0.,0.,1.,); r12 = Point3D(1.,0.,1.); 
e1hat = Edge3D(r1 = r11,r2 = r12)

# return second edge 
r21 = Point3D(0.,0.,1.,); r22 = Point3D(0.,1.,1.); 
e2 = Edge3D(r1 = r21,r2 = r22)

h = 1; 

In [ ]:
I35integrand(y) = y[1]*f3(1,sqrt(1+y[1]^2))
I35testvalnum = hcubature(y -> I35integrand(y), (0,), (1,);rtol=1e-10,atol=1e-10)[1]
smin = 0.; smax = 1.; h = 1.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
I35testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I35 = ", I35testvalnum)
println(" ana. I35 = ", I35testvalana)
@test I35testvalnum ≈ I35testvalana

I36integrand(y) = y[1]*f3(1,sqrt(y[1]^2))
cutoff = 1e-6
I36testvalnum = hcubature(y -> I36integrand(y), (cutoff,), (1,);rtol=1e-10,atol=1e-10)[1]
smin = cutoff; smax = 1.; h = 1.; R0 = 0.  
LB = (smin,h,R0); UB = (smax,h,R0)
I36testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I36 = ", I36testvalnum)
println(" ana. I36 = ", I36testvalana)
@test I36testvalnum ≈ I36testvalana

I37integrand(x) = x[1]*f3(1,sqrt(1+x[1]^2))
I37testvalnum = hcubature(x -> I37integrand(x), (0,), (1,);rtol=1e-10,atol=1e-10)[1]
smin = 0.; smax = 1.; h = 1.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
I37testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I37 = ", I37testvalnum)
println(" ana. I37 = ", I37testvalana)
@test I37testvalnum ≈ I37testvalana

I38integrand(x) = x[1]*f3(1,sqrt(x[1]^2))
I38testvalnum = hcubature(x -> I38integrand(x), (0,), (1,);rtol=1e-10,atol=1e-10)[1]
smin = 0.; smax = 1.; h = 1.; R0 = 0.  
LB = (smin,h,R0); UB = (smax,h,R0)
I38testvalana = @eval_bound_diff I13 (LB, UB)
println(" num. I38 = ", I38testvalnum)
println(" ana. I38 = ", I38testvalana)
@test I38testvalnum ≈ I38testvalana

I31integrand(lmbd) = lmbd[1]*lmbd[2]/ sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
I31testvalnum = hcubature(lmbd -> I31integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
svec = [0,0,1] 
I31testvalana = norm(e1hat.r2-svec)^2*I35testvalana 
I31testvalana += -norm(e1hat.r1-svec)^2*I36testvalana  
I31testvalana += norm(e2.r2-svec)^2*I37testvalana
I31testvalana += -norm(e2.r1-svec)^2*I38testvalana
println(" num. I31 = ", I31testvalnum)
println(" ana. I31 = ", I31testvalana)
@test I31testvalnum ≈ I31testvalana

I32integrand(lmbd) = lmbd[1]/ sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
I32testvalnum = hcubature(lmbd -> I32integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
h = 1.; svec = [0,0,1]
# upper bound for Edge-2
smin = 1.; smax = 2.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term1 = norm(e1hat.r2-svec)^2*factor
# lower bound for Edge-2
smin = 1.; smax = 2.; R0 = 0.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term2 = -norm(e1hat.r1-svec)^2*factor 
# upper bound for Edge-1
smin = 0.; smax = 1.; R0 = 2.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e2.r2-svec,e1hat.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term3 = dot(e2.r2-svec,e2.tau)*factor 
# lower bound for Edge-1
smin = 0.; smax = 1.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e2.r1-svec,e1hat.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term4 = -dot(e2.r1-svec,e2.tau)*factor
I32testvalana = term1 + term2 + term3 + term4  
println(" num. I32 = ", I32testvalnum)
println(" ana. I32 = ", I32testvalana)
@test I32testvalnum ≈ I32testvalana

I33integrand(lmbd) = lmbd[2]/ sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
I33testvalnum = hcubature(lmbd -> I33integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
h = 1.; svec = [0,0,1]
# upper bound for Edge-1
smin = 0.; smax = 1.; R0 = 2.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term1 = norm(e2.r2-svec)^2*factor 
# lower bound for Edge-1
smin = 0.; smax = 1.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
term2 = -norm(e2.r1-svec)^2*factor 
# upper bound for Edge-2
smin = 1.; smax = 2.; R0 = 1.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e1hat.r2-svec,e2.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term3 = dot(e1hat.r2-svec,e1hat.tau)*factor 
# lower bound for Edge-2
smin = 1.; smax = 2.; R0 = 0.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = @eval_bound_diff I10 (LB, UB)
factor *= dot(e1hat.r1-svec,e2.tau)
factor += @eval_bound_diff I11 (LB, UB) 
term4 = -dot(e1hat.r1-svec,e1hat.tau)*factor
I33testvalana = term1 + term2 + term3 + term4 
println(" num. I33 = ", I33testvalnum)
println(" ana. I33 = ", I33testvalana)
@test I33testvalnum ≈ I33testvalana

I34integrand(lmbd) = 1 / sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
I34testvalnum = hcubature(lmbd -> I34integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
h = 1.; svec = [0,0,1]
I34testvalana = dot(e1hat.r2-svec,e1hat.tau)*R(e1hat.r2,e2,h)
I34testvalana += -dot(e1hat.r1-svec,e1hat.tau)*R(e1hat.r1,e2,h)
I34testvalana += dot(e2.r2-svec,e2.tau)*R(e2.r2,e1hat,h)
I34testvalana += -dot(e2.r1-svec,e2.tau)*R(e2.r1,e1hat,h)
println(" num. I34 = ", I34testvalnum)
println(" ana. I34 = ", I34testvalana)
@test I34testvalnum ≈ I34testvalana

Qtestintegrand(lmbd) = (1-lmbd[1])*(1-lmbd[2])/ sqrt(lmbd[1]^2 + lmbd[2]^2 +1) 
Qtestvalnum = hcubature(lmbd -> Qtestintegrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
Qtestvalana = I31testvalana - I32testvalana - I33testvalana + I34testvalana
println(" num. Qnum = ", Qtestvalnum)
println(" ana. Qana = ", Qtestvalana)
@test Qtestvalnum ≈ Qtestvalana

### Test Case of Coinciding Edges: Unit Edge Along x-Axis

Assume 
* Edge-1 to extend along the Point-1 ${\mathbf r}^1 = (0,0,0)$ to the Point-2 ${\mathbf r}^2 = (1,0,0)$. The destination and source position vectors along the edge can be written ${\mathbf r} = (x,0,0)$ and ${\mathbf r}_p = (x_p,0,0)$, respectively. Thus $\|{\mathbf r} - {\mathbf r}_p \| = \sqrt{(x-x_p)^2}$. The position vector along the edge is given by ${\mathbf \tau}_1 = {\mathbf e}_1 = (1,0,0)$. 
* We understand that $\theta_1({\mathbf r}^1_2) = 0$, and that that the term involving $I_{27}$ cancels in the computation of $Q^{\phi_1,\theta_1}(\ell_1,\ell_2)$. 

We thus have that 

<b>Computation of $I_{27}$ - Fail to understand the symbolic computation - Fails symbolically (unsure how to fix)</b>

$$
I_{27} = \int_0^1 dx \int_0^1 dx_p \frac{x}{\sqrt{(x - x_p)^2}} = 5.05741 ... 
$$

<b>Computation of $I_{41}$ - Fails symbolically (unsure how to fix)</b>

$$
I_{41} = \int_0^1 dx \int_0^1 dx_p \frac{x \, x_p}{\sqrt{(x - x_p)^2}} = 2.9563 ... 
$$

<b>Computation of $Q^{\phi_1,\theta_1}(\ell_1,\ell_2)$ - Fails symbolically (unsure how to fix)</b>

$$
Q^{\phi_1,\theta_1}(\ell_1,\ell_2) = \int_0^1 dx \int_0^1 dx_p \frac{\varphi_1({\mathbf r}) \, \theta_1({\mathbf r}_p)}{\|{\mathbf r} - {\mathbf r}_p \|} = \int_0^1 dx \int_0^1 dx_p \frac{(1 - x) \, (1 - x_p)}{\sqrt{(x - x_p)^2}} = 2.9563 ... \text{(fully numerical)} = I_{41} - \theta_1({\mathbf r}^1_2) I_{27} = I_{41} - 0 * I_{27} = 2.9563 ... \text{(analytically)}\, . 
$$

In [ ]:
# return first edge 
r11 = Point3D(0.,0.,0.,); r12 = Point3D(1.,0.,0.); 
e1 = Edge3D(r1 = r11,r2 = r12)
e2 = deepcopy(e1)

In [ ]:
term1 = (-1)*(-1*Rphi1(e1.r1,e2))
term2 = 0*Rphi1(e2.r2,e1) 
term3 = -Rphi1(e2.r1,e1) 
[term1, term2, term3]

In [ ]:
I27integrand(lmbd) = lmbd[1] / abs(lmbd[1] - lmbd[2])
I27testvalnum = hcubature(lmbd -> I27integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
term1 = -.5*norm(e1.r1-e1.r2)*I3(e1.r1,e2)
println("   I27ana term-1 = ", term1)
term2 = -.5*dot(e2.r2-e1.r2,e2.tau)*Rphi1(e2.r2,e1)
println("   I27ana term-2 = ", term2)
term3 = .5*dot(e2.r1-e1.r2,e2.tau)*Rphi1(e2.r1,e1)
println("   I27ana term-3 = ", term3)
println(" num. I27 = ", I27testvalnum)

I41integrand(lmbd) = lmbd[1]*lmbd[2]/ abs(lmbd[1] - lmbd[2])
I41testvalnum = hcubature(lmbd -> I41integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
term1 = -.5*dot(e1.r1-e1.r2,e2.tau)*(-Rphi1(e1.r1,e2))
println("   I41ana term-1 = ", term1)
term2 = -dot(e2.r2-e1.r2,e2.tau)*Rphi1(e2.r2,e1)
println("   I41ana term-2 = ", term2)
term3 = dot(e2.r1-e1.r2,e2.tau)*Rphi1(e2.r1,e1)
println("   I41ana term-3 = ", term3)
println(" num. I41 = ", I41testvalnum)

Qtestintegrand(lmbd) = (1-lmbd[1])*(1-lmbd[2])/ abs(lmbd[1] - lmbd[2])
Qtestvalnum = hcubature(lmbd -> Qtestintegrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]  
println(" num. Qnum = ", Qtestvalnum)


### Test Case of Parallel Edges: Unit Edge Along x-Axis (see also symbolic computation)

Assume (this test case is again symmetrical wrt the cooordinates $x$ and $y$)

* Edge-1 to extend along the x-Axis from the Point-1 ${\mathbf r}^1 = (0,0,0)$ to the Point-2 ${\mathbf r}^2 = (1,0,0)$. The destination position vector along Edge-1 can be written ${\mathbf r} = (x,0,0)$ for $0 \leq x \leq 1$. The direction vectors along Edge-1 are given by ${\mathbf \tau}_1 = {\mathbf e}_1 = (1,0,0)$;  
* Edge-2 to be Edge-1 shifted along the $z$-axis be a distance equal 1. That is, assume Edge-2 to extend along the x-Axis from the Point-1 ${\mathbf r}^1 = (0,0,1)$ to the Point-2 ${\mathbf r}^2 = (1,0,1)$.  The source position vector along Edge-2 can be written as ${\mathbf r}_p = (x_p,0,1)$ for $0 \leq x_p \leq 1$. The direction vectors along Edge-2 are given by ${\mathbf \tau}_2 = {\mathbf e}_2 = (1,0,0)$; 
* the distance between Edge-1 (source) - Edge-2 (destination) can be written as $\|{\mathbf r} - {\mathbf r}_p \| = \sqrt{(x-x_p)^2+1)}$.  

We thus have that 

<b>Computation of $I_{32}$</b>

$$
I_{32} = \int_0^1 dx \int_0^1 dx_p \frac{x}{\sqrt{(x - x_p)^2+1}} = 0.4671 ... 
$$

<b>Computation of $I_{36}$</b>

$$
I_{36} = \int_0^1 dx_p \, (x_p - 1) \, f_3(1,x_p) = \int_0^1 dx_p \, x_p \, f_3(1,x_p) - \int_0^1 dx_p \, f_3(1,x_p) = -0.1192 ... 
$$

<b>Computation of $I_{37}$</b>

$$
I_{37} = \int_0^1 dx \, (x - 1) \, f_3(1,1-x) = -0.1094 ... 
$$

<b>Computation of $I_{38}$</b> Identical to $I_{36}$ with $x$ replaced by $x_p$ 

$$
I_{38} = \int_0^1 dx \, (x - 1) \, f_3(1,x) = -0.1192 ... 
$$

<b>Computation of $I_{47}$</b>

$$
I_{47} = \int_0^1 dx \int_0^1 dx_p \frac{x \, x_p}{\sqrt{(x - x_p)^2+1}} = 0.2384 ... 
$$

<b>Computation of $Q^{\phi_1,\theta_1}(\ell_1,\ell_2)$</b>

$$
Q^{\phi_1,\theta_1}(\ell_1,\ell_2) = \int_0^1 dx \int_0^1 dx_p \frac{\varphi_1({\mathbf r}) \, \theta_1({\mathbf r}_p)}{\|{\mathbf r} - {\mathbf r}_p \|} = \int_0^1 dx \int_0^1 dx_p \frac{(1 - x) \, (1 - x_p)}{\sqrt{(x - x_p)^2+1}} = 0.2384 ... \text{(fully numerical)} = I_{47} - \theta_1(\hat{{\mathbf r}}^1_2) I_{32} = 0.2384 ... \text{(analytically)} \, . 
$$

In [ ]:
# return first edge 
r11 = Point3D(0.,0.,0.,); r12 = Point3D(1.,0.,0.); 
e1 = Edge3D(r1 = r11,r2 = r12)

# return secon edge 
r21 = Point3D(0.,0.,1.,); r22 = Point3D(1.,0.,1.); 
e2 = Edge3D(r1 = r21,r2 = r22)

e1hat = deepcopy(e2);

In [ ]:
I36integrand(lmbd) = (lmbd[1]-1) * f3(1.,lmbd[1]) 
I36testvalnum = hcubature(lmbd -> I36integrand(lmbd), (0,), (1,);rtol=1e-6,atol=1e-6)[1]
I36integrandterm1(lmbd) = lmbd[1] * f3(1.,lmbd[1]) 
I36testvalnumterm1 = hcubature(lmbd -> I36integrandterm1(lmbd), (0,), (1,);rtol=1e-6,atol=1e-6)[1]
I36integrandterm2(lmbd) = -1 * f3(1.,lmbd[1]) 
I36testvalnumterm2 = hcubature(lmbd -> I36integrandterm2(lmbd), (0,), (1,);rtol=1e-6,atol=1e-6)[1]
smin = 0.; smax = 1.; h = 1.; R0 = 0.  
factor = myI12delta(smin,smax,h,R0)
factor *= -1.  
factor += myI13delta(smin,smax,h,R0) 
I36testvalana = factor
println(" ana. I36 = ", I36testvalana)
println("   num. I36term1 = ", I36testvalnumterm1)
println("   num. I36term2 = ", I36testvalnumterm2)
println(" num. I36 = ", I36testvalnum)

I37integrand(lmbd) = (lmbd[1]-1) * f3(1.,1-lmbd[1]) 
I37testvalnum = hcubature(lmbd -> I37integrand(lmbd), (0,), (1,);rtol=1e-6,atol=1e-6)[1]
smin = -1.; smax = 0.; h = 1.; R0 = 0.  
I37testvalana = myI13delta(smin,smax,h,R0)
println(" ana. I37 = ", I37testvalana)
println(" num. I37 = ", I37testvalnum)

I38integrand(lmbd) = (lmbd[1]-1) * f3(1.,lmbd[1]) 
I38testvalnum = hcubature(lmbd -> I38integrand(lmbd), (0,), (1,);rtol=1e-6,atol=1e-6)[1]
println(" num. I38 = ", I38testvalnum)

In [ ]:
I32integrand(lmbd) = -lmbd[1] / sqrt((lmbd[1] - lmbd[2])^2+1) 
I32testvalnum = hcubature(lmbd -> I32integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
# term-1
smin = 0.; smax = 1.; h = 1.; R0 = 0.  
LB = (smin,h,R0); UB = (smax,h,R0)
term1 = -norm(e2.r2-e2.r1)*myI10delta(smin,smax,h,R0) 
println("   I32ana term-1 = ", term1)
# term-2
smin = -1.; smax = 0.; h = 1.; R0 = 0.  
LB = (smin,h,R0); UB = (smax,h,R0)
factor = h*myI11delta(smin,smax,h,R0)+dot(e2.r1-e1hat.r2,e1.tau)*myI10delta(smin,smax,h,R0)
term2 = -dot(e2.r2-e1hat.r2,e2.tau)*factor
println("   I32ana term-2 = ", term2)
# term-3
smin = 0.; smax = 1.; h = 1.; R0 = 0.  
LB = (smin,h,R0); UB = (smax,h,R0) 
factor = h*myI11delta(smin,smax,h,R0)+dot(e2.r1-e1hat.r2,e1.tau)*myI10delta(smin,smax,h,R0)
term3 = -dot(e2.r1-e1hat.r2,e2.tau)*factor 
println("   I32ana term-3 = ", term3)
I32testvalana = term1 + term2 + term3 
println(" ana. I32 = ", I32testvalana)
println(" num. I32 = ", I32testvalnum)

I47integrand(lmbd) = lmbd[1]*lmbd[2]/ sqrt((lmbd[1] - lmbd[2])^2+1)
I47testvalnum = hcubature(lmbd -> I47integrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]
I47testvalana = -norm(e1.r2-e1.r1)*I36testvalana 
I47testvalana += dot(e2.r2-e1hat.r2,e1.tau)^2*I37testvalana 
I47testvalana += dot(e2.r1-e1hat.r2,e1.tau)^2*I38testvalana  
println(" ana. I47 = ", I47testvalana)
println(" num. I47 = ", I47testvalnum)

Qtestintegrand(lmbd) = (1-lmbd[1])*(1-lmbd[2])/ sqrt((lmbd[1] - lmbd[2])^2+1) 
Qtestvalnum = hcubature(lmbd -> Qtestintegrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]  
println(" num. Qnum = ", Qtestvalnum)

### Define Sample data for intersecting edges 

In [ ]:
# return first edge 
r11 = Point3D(1.,1.,0.,); r12 = Point3D(2.,2.,0.); 
e1 = Edge3D(r1 = r11,r2 = r12)

# return second edge 
r21 = Point3D(1.,0.,0.,); r22 = Point3D(2.,0.,0.);
e2 = Edge3D(r1 = r21,r2 = r22)

# return EdgePair
edgePair = EdgePair(e1 = e1, e2 = e2)

dist = distanceEdgePair(edgePair)

dir = directionEdgePair(edgePair)

# return intersecting edge pair 
interEdgePair = InterEdgePair(edgePair = edgePair)

### Define Sample data for crossing edges  

In [ ]:
# return first edge 
r11 = Point3D(0.,0.,0.,); r12 = Point3D(1.,0.,0.); 
e1 = Edge3D(r1 = r11,r2 = r12)

# return second edge 
r21 = Point3D(0.,0.,1.,); r22 = Point3D(0.,1.,1.); 
e2 = Edge3D(r1 = r21,r2 = r22)

# return EdgePair
edgePair = EdgePair(e1 = e1, e2 = e2)

# return intersecting edge pair 
crossEdgePair1 = CrossEdgePair(edgePair = edgePair)

######################################################

# return first edge 
r11 = Point3D(1.,0.,0.,); r12 = Point3D(2.,0.,0.); 
e1 = Edge3D(r1 = r11,r2 = r12)

# return second edge 
r21 = Point3D(0.,0.,1.,); r22 = Point3D(0.,1.,1.); 
e2 = Edge3D(r1 = r21,r2 = r22)

# return EdgePair
edgePair = EdgePair(e1 = e1, e2 = e2)

# return intersecting edge pair 
crossEdgePair2 = CrossEdgePair(edgePair = edgePair)

######################################################

# return first edge 
r11 = Point3D(1.,1.,1.,); r12 = Point3D(2.,2.,1.); 
e1 = Edge3D(r1 = r11,r2 = r12)

# return second edge 
r21 = Point3D(1.,0.,0.,); r22 = Point3D(2.,0.,0.); 
e2 = Edge3D(r1 = r21,r2 = r22)

# return EdgePair
edgePair = EdgePair(e1 = e1, e2 = e2)

# return intersecting edge pair 
crossEdgePair2 = CrossEdgePair(edgePair = edgePair)

crossEdgePair2.e1hat 

### Define Sample data for parallel edges  

### Define Sample data for coinciding edges  

In [ ]:
# define first edege 
r11 = Point3D(0.,0.,0.,); r12 = Point3D(1.,0.,0.); 
e1 = Edge3D(r1 = r11,r2 = r12)

# define second edge 
e2 = deepcopy(e1)

edgePair = EdgePair(e1=e1, e2=e2)

coinEdgePair1 = CoinEdgePair(edgePair = edgePair)

##############################################################

# define first edege 
r11 = Point3D(1.,1.,1.,); r12 = Point3D(2.,2.,1.); 
e1 = Edge3D(r1 = r11,r2 = r12)

# define second edge 
e2 = deepcopy(e1)

edgePair = EdgePair(e1=e1, e2=e2)

coinEdgePair2 = CoinEdgePair(edgePair = edgePair);

coinEdgePair1.edgePair.e1(coinEdgePair1.edgePair.e1.r2)[1]

## Section 3: Integral Computations 

### Section 1.3: Crossing Lines 

In [ ]:
function I31(crossEdgePair::CrossEdgePair)
    edgePair = crossEdgePair.edgePair
    e1hat    = crossEdgePair.e1hat 
    e2       = edgePair.e2 
    svec     = crossEdgePair.svec 
    ans =   norm(e1hat.r2-svec)^2/e1.len*I35(crossEdgePair)
    ans += -norm(e1hat.r2-svec)^2/e1.len*I36(crossEdgePair)
    ans +=  norm(e2.r2-svec)^2/e2.len*I37(crossEdgePair)
    ans += -norm(e2.r1-svec)^2/e2.len*I38(crossEdgePair)
    return ans  
end

function I31integrand(lmbd, crossEdgePair::CrossEdgePair)
    edgePair = crossEdgePair.edgePair
    e1hat    = crossEdgePair.e1hat 
    e2       = edgePair.e2 
    h        = crossEdgePair.h 
    r        = e1hat.r1+lmbd[1]*e1hat.tau 
    rp       = e2.r1+lmbd[2]*e2.tau
    return dot(r,e1hat.e)*dot(rp,e2.e)/sqrt(h^2+norm(r-rp)^2)
end

function I32(crossEdgePair::CrossEdgePair)
    return 0. 
end

function I32integrand(lmbd, crossEdgePair::CrossEdgePair)
    edgePair = crossEdgePair.edgePair
    e1hat    = crossEdgePair.e1hat 
    e2       = edgePair.e2 
    h        = crossEdgePair.h 
    svec     = crossEdgePair.svec 
    r        = e1hat.r1+lmbd[1]*e1hat.tau 
    rp       = e2.r1+lmbd[2]*e2.tau
    return dot(r-svec,e1hat.e)/sqrt(h^2+norm(r-rp)^2)    
end

function I33(crossEdgePair::CrossEdgePair)
    return 0.
end

function I33integrand(lmbd, crossEdgePair::CrossEdgePair) 
    edgePair = crossEdgePair.edgePair
    e1hat    = crossEdgePair.e1hat 
    e2       = edgePair.e2 
    h        = crossEdgePair.h 
    svec     = crossEdgePair.svec 
    r        = e1hat.r1+lmbd[1]*e1hat.tau 
    rp       = e2.r1+lmbd[2]*e2.tau
    return dot(rp-svec,e2.e)/sqrt(h^2+norm(r-rp)^2)
end

function I34(crossEdgePair::CrossEdgePair)
    edgePair = crossEdgePair.edgePair
    e1hat    = crossEdgePair.e1hat 
    e2       = edgePair.e2 
    h        = crossEdgePair.h 
    ans = dot(e1hat.r2,e1hat.tau)*R(e1hat.r2,e2,h)
    ans += -dot(e1hat.r1,e1hat.tau)*R(e1hat.r1,e2,h)
    ans += dot(e2.r2,e2.tau)*R(e2.r2,e1hat,h)
    ans += -dot(e2.r1,e2.tau)*R(e2.r1,e1hat,h)
    return ans 
end

function I34integrand(lmbd, crossEdgePair::CrossEdgePair)
    edgePair = crossEdgePair.edgePair
    e1hat    = crossEdgePair.e1hat 
    e2       = edgePair.e2 
    h        = crossEdgePair.h 
    r        = e1hat.r1+lmbd[1]*e1hat.tau 
    rp       = e2.r1+lmbd[2]*e2.tau
    return 1/sqrt(h^2+norm(r-rp)^2)    
end

"""
   I35(crossEdge3DPair)

Returns point(e1.r2) - edge(e2) interaction
"""
function I35(crossEdge3DPair)
    
    function I35aux(s,crossEdge3DPair)
        h     = crossEdge3DPair.h 
        R0    = crossEdge3DPair.R0 
        tau2  = crossEdge3DPair.e2.tau 
        r2hat = crossEdge3DPair.r2hat 
        svec  = crossEdge3DPair.svec  
        ans  = h*I13(s,h,R0)
        ans += dot(r2hat-svec,tau2)*I12(s,h,R0)
        return ans; 
    end 
 
    e2    = crossEdge3DPair.e2
    r2hat = crossEdge3DPair.r2hat
    smin  = dot(e2.tau,e2.r1-r2hat) 
    smax  = dot(e2.tau,e2.r2-r2hat)
    ans   = (I35aux(smax,crossEdge3DPair) - I35aux(smin,crossEdge3DPair))/e2.len
    
    return ans  
end

function I35integrand(lmbd2,crossEdge3DPair)
    h     = crossEdge3DPair.h   
    e2    = crossEdge3DPair.e2 
    r2hat = crossEdge3DPair.r2hat 
    svec  = crossEdge3DPair.svec
    rp    = e2.r1+lmbd2[1]*e2.tau
    return dot(rp-svec,e2.e)*f3(h,norm(r2hat-rp))
end
    
"""
   I36(crossEdge3DPair)

Returns point(e1.r1) - edge(e2) interaction
"""    
function I36(crossEdge3DPair)
    
    function I36aux(s,crossEdge3DPair)
        h     = crossEdge3DPair.h 
        R0    = crossEdge3DPair.R0 
        tau2  = crossEdge3DPair.e2.tau 
        r1hat = crossEdge3DPair.r1hat 
        svec  = crossEdge3DPair.svec  
        ans  = h*I13(s,h,R0)
        ans += dot(r1hat-svec,tau2)*I12(s,h,R0)
        return ans; 
    end 
 
    e2    = crossEdge3DPair.e2
    r1hat = crossEdge3DPair.r1hat
    smin  = dot(e2.tau,e2.r1-r1hat) 
    smax  = dot(e2.tau,e2.r2-r1hat)
    ans   = (I36aux(smax,crossEdge3DPair) - I36aux(smin,crossEdge3DPair))/e2.len
    
    return ans  
end

function I36integrand(lmbd2,crossEdge3DPair)
    h     = crossEdge3DPair.h   
    e2    = crossEdge3DPair.e2 
    r1hat = crossEdge3DPair.r1hat 
    svec  = crossEdge3DPair.svec
    rp    = e2.r1+lmbd2[1]*e2.tau
    return dot(rp-svec,e2.e)*f3(h,norm(r1hat-rp))
end
 
"""
   I37(crossEdge3DPair)

Returns point(e2.r1) - edge(e1) interaction
"""    
function I37(crossEdge3DPair)
    
    function I37aux(s,crossEdge3DPair)
        h     = crossEdge3DPair.h 
        R0    = crossEdge3DPair.R0 
        tau1  = crossEdge3DPair.e1.tau 
        r1hat = crossEdge3DPair.r1hat 
        svec  = crossEdge3DPair.svec  
        ans  = h*I13(s,h,R0)
        ans += dot(e2.r2-svec,tau1)*I12(s,h,R0)
        return ans; 
    end 
 
    e1    = crossEdge3DPair.e1
    e2    = crossEdge3DPair.e2
    r1hat = crossEdge3DPair.r1hat
    r2hat = crossEdge3DPair.r2hat
    smin  = dot(e1.tau,r1hat-e2.r2) 
    smax  = dot(e1.tau,r2hat-e2.r2)
    ans   = (I37aux(smax,crossEdge3DPair) - I37aux(smin,crossEdge3DPair))/norm(e1.len)
    
    return ans  
end

function I37integrand(lmbd1,crossEdge3DPair)
    h     = crossEdge3DPair.h   
    e1    = crossEdge3DPair.e1 
    r1hat = crossEdge3DPair.r1hat 
    svec  = crossEdge3DPair.svec
    r     = r1hat+lmbd1[1]*e1.tau
    return dot(r-svec,e1.e)*f3(h,norm(r-e2.r2))
end
            
"""
   I38(crossEdge3DPair)

Returns point(e2.r2) - edge(e1) interaction
"""        
function I38(crossEdge3DPair)
    
    function I38aux(s,crossEdge3DPair)
        h     = crossEdge3DPair.h 
        R0    = crossEdge3DPair.R0 
        tau1  = crossEdge3DPair.e1.tau 
        r1hat = crossEdge3DPair.r1hat 
        svec  = crossEdge3DPair.svec  
        ans  = h*I13(s,h,R0)
        ans += dot(e2.r1-svec,tau1)*I12(s,h,R0)
        return ans; 
    end 

    e1    = crossEdge3DPair.e1
    e2    = crossEdge3DPair.e2
    r1hat = crossEdge3DPair.r1hat
    r2hat = crossEdge3DPair.r2hat        
    smin  = dot(e1.tau,r1hat-e2.r1) 
    smax  = dot(e1.tau,r2hat-e2.r1)
    ans   = (I38aux(smax,crossEdge3DPair) - I38aux(smin,crossEdge3DPair))/norm(e1.len)
    
    return ans  
end    

function I38integrand(lmbd1,crossEdge3DPair)
    h     = crossEdge3DPair.h   
    e1    = crossEdge3DPair.e1 
    r1hat = crossEdge3DPair.r1hat 
    svec  = crossEdge3DPair.svec
    r     = r1hat+lmbd1[1]*e1.tau
    return dot(r-svec,e1.e)*f3(h,norm(r-e2.r1))
end
    
function Qphi1theta1(crossEdgePair::CrossEdgePair)
    edgePair = crossEdgePair.edgePair
    svec     = crossEdgePair.svec 
    e1       = edgePair.e1 
    e2       = edge
    ans  = I31(crossEdgePair)-e2(svec)[1]*I32(crossEdgePair)
    ans += -e1hat(svec)[1]*I33(crossEdgePair)+e1hat(svec)[1]*e2(svec)[2]*I34(crossEdgePair)
    return ans 
end

function Qphi1theta1integrand(lmbd, crossEdgePair::CrossEdgePair)
    edgePair = crossEdgePair.edgePair
    e1hat    = crossEdgePair.e1hat 
    e2       = edgePair.e2 
    h        = crossEdgePair.h 
    r        = e1hat.r1+lmbd[1]*e1hat.tau 
    rp       = e2.r1+lmbd[2]*e2.tau
    return dot(r-e1hat.r2,e1hat.e)*dot(rp-e2.r2,e2.e)/sqrt(h^2+norm(r-rp)^2)
end;

In [ ]:
function I36integrand(lmbd2,crossEdge3DPair)
    h     = crossEdge3DPair.h   
    e2    = crossEdge3DPair.e2
    r1hat = crossEdge3DPair.r1hat 
    svec  = crossEdge3DPair.svec
    rp    = e2.r1+lmbd2[1]*e2.tau
    ans1   = rp-svec
    ans2   = e2.e
    ans3   = dot(rp-svec,e2.e)
    #println("ans1 is ", ans1)
    #println("ans2 is ", ans2)
    #println("ans3 is ", ans3)
    return dot(rp-svec,e2.tau)*f3(h,norm(r1hat-rp))
end

I36integrand(0.,crossEdgePair1)

In [ ]:
len1 = crossEdgePair1.e1hat.len; len2 = crossEdgePair1.edgePair.e2.len; 
I36valnum = hcubature(lmbd -> I36integrand(lmbd,crossEdgePair1), (0,), (len2,);rtol=1e-10,atol=1e-10)[1] 

In [ ]:
I37(crossEdgePair2)

In [ ]:
len1 = crossEdgePair2.e1hat.len; len2 = crossEdgePair2.edgePair.e2.len; 
I37valnum = hcubature(lmbd -> I37integrand(lmbd,crossEdgePair2), (0,), (len1,);rtol=1e-10,atol=1e-10)[1]  

In [ ]:
I38(crossEdgePair2)

In [ ]:
len1 = crossEdgePair2.e1hat.len; len2 = crossEdgePair2.edgePair.e2.len; 
I38valnum = hcubature(lmbd -> I38integrand(lmbd,crossEdgePair2), (0,), (len1,);rtol=1e-10,atol=1e-10)[1]  

In [ ]:
len1 = crossEdgePair2.e1hat.len; len2 = crossEdgePair2.edgePair.e2.len; 

I34val = I34(crossEdgePair2)
I35val = I35(crossEdgePair2)
I34valnum = hcubature(lmbd -> I34integrand(lmbd,crossEdgePair2), (0,0), (len1,len2);rtol=1e-10,atol=1e-10)[1] 
I35valnum = hcubature(lmbd -> I35integrand(lmbd,crossEdgePair2), (0,), (len2,);rtol=1e-10,atol=1e-10)[1] 

@test I34val ≈ I34valnum 
@test I35val ≈ I35valnum

In [ ]:
function Itestintegrand(lmbd) 
    h     = 1.;                       # distance between edges
    e1    = Point3D(1.,0.,0.)         # unit vector along x-direction
    e2    = Point3D(0.,1.,0.)         # unit vector along y-direction
    rhat2 = Point3D(1.,0.,1.)         # end point projected first edge
    r22   = Point3D(0.,1.,1.)         # end point second edge 
    r     = Point3D(lmbd[1],0.,1.)    # along projected-edge-1
    rp    = Point3D(0.,lmbd[2],1.)    # along edge-2
    return dot(r-rhat2,e1)*dot(rp-r22,e2)/sqrt(h^2+norm(r-rp)^2)
end 

Itestvalnum = hcubature(lmbd -> Itestintegrand(lmbd), (0,0), (1,1);rtol=1e-10,atol=1e-10)[1]  


In [ ]:
Qphi1theta1integrand([.5,.5],crossEdgePair1)

In [ ]:
Qphi1theta1valnum = hcubature(lmbd -> Qphi1theta1integrand(lmbd,crossEdgePair1), (0,0), (len1,len2);rtol=1e-10,atol=1e-10)[1]


In [ ]:
function Qphi1theta1(interEdgePair::InterEdgePair)
    edgePair = interEdgePair.edgePair
    svec     = interEdgePair.svec 
    e1       = edgePair.e1 
    e2       = edgePair.e2
    ans = I25(interEdgePair)-e1(svec)[1]*I26(interEdgePair)
    ans += -e2(svec)[1]*I27(interEdgePair)+e1(svec)[1]*e2(svec)[1]*I28(dgePair)
    return ans 
end

function Qphi1theta1integrand(lmbd,interEdgePair::InterEdgePair)
    edgePair = interEdgePair.edgePair
    e1 = edgePair.e1 
    e2 = edgePair.e2    
    r  = e1.r1+lmbd[1]*e1.tau 
    rp = e2.r1+lmbd[2]*e2.tau
    return (e1(r)[1])*(e2(rp)[1])/norm(r-rp)
end;

In [ ]:
I38(crossEdgePair2)

In [ ]:
I38valnum = hcubature(lmbd -> I38integrand(lmbd[1],crossEdgePair2), (0,), (e1.len,);rtol=1e-10,atol=1e-10)[1]

In [ ]:
I37(crossEdge3DPair)

In [ ]:
I37valnum = hcubature(lmbd -> I37integrand(lmbd[1],crossEdge3DPair), (0,), (e1.len,);rtol=1e-10,atol=1e-10)[1]

In [ ]:
methods(I37integrand)

In [ ]:
I35val = I35(crossEdge3DPair) 

I35valnum = hcubature(lmbd -> I35integrand(lmbd[1],crossEdge3DPair), (0,), (e2.len,);rtol=1e-10,atol=1e-10)[1]

@test I35val ≈ I35valnum

In [ ]:
I36val = I36(crossEdge3DPair)

In [ ]:
methods(I36integrand)

In [ ]:
I36valnum = hcubature(lmbd -> I36integrand(lmbd[1],crossEdge3DPair), (0,), (e2.len,);rtol=1e-10,atol=1e-10)[1]


### Testing of Intersecting Edge - Edge Interaction 

In [ ]:
# define a first edge
r11 = Point3D(1.,1.,0.,); r12 = Point3D(2.,2.,0.); len = norm(r12-r11)
e1 = Edge3D(r1 = r11, r2 = r12) 

# define a second edge
r21 = Point3D(1.,0.,0.,); r22 = Point3D(2.,0.,0.); len = norm(r22-r21)
e2 = Edge3D(r1 = r21, r2 = r22)

# define edge pair 
edgePair = EdgePair(e1 = e1, e2 = e2);

# define intersecting edge pair (change name output to interEdgePair)
interEdgePair = InterEdgePair(edgePair = edgePair);

In [ ]:
len1 = interEdgePair.edgePair.e1.len; len2 = interEdgePair.edgePair.e2.len
I25val = I25(interEdgePair)
I26val = I26(interEdgePair)
I27val = I27(interEdgePair)
I28val = I28(interEdgePair)
Qphi1theta1val = Qphi1theta1(interEdgePair)

I25valnum = hcubature(lmbd -> I25integrand(lmbd,interEdgePair), (0,0), (len1,len2);rtol=1e-10,atol=1e-10)[1]  
I26valnum = hcubature(lmbd -> I26integrand(lmbd,interEdgePair), (0,0), (len1,len2);rtol=1e-10,atol=1e-10)[1]  
I27valnum = hcubature(lmbd -> I27integrand(lmbd,interEdgePair), (0,0), (len1,len2);rtol=1e-10,atol=1e-10)[1]  
I28valnum = hcubature(lmbd -> I28integrand(lmbd,interEdgePair), (0,0), (len1,len2);rtol=1e-10,atol=1e-10)[1]
Qphi1theta1valnum = hcubature(lmbd -> Qphi1theta1integrand(lmbd,interEdgePair), (0,0), (len1,len2);rtol=1e-10,atol=1e-10)[1]

@test I25val ≈ I25valnum
@test I26val ≈ I26valnum
@test I27val ≈ I27valnum
@test I28val ≈ I28valnum
@test Qphi1theta1val ≈ Qphi1theta1valnum

In [ ]:
@code_warntype I25(myInterEdgePair)

### Section 3.3: Case of Parallel Lines

In [ ]:
function I32(paralEdgePair::ParalEdgePair)
    return 0. 
end

function I32integrand(lmbd,paralEdgePair::ParalEdgePair)
    return 0.
end

function I36(paralEdgePair::ParalEdgePair)
    
    function I36aux(s,paralEdge3DPair)
        ans = 0.
        return ans; 
    end
    
    return 0. 
end

function I36integrand(lmbd,paralEdgePair::ParalEdgePair)
    return 0.
end

function I37(paralEdgePair::ParalEdgePair)
        
    function I37aux(s,paralEdge3DPair)
        ans = 0.
        return ans; 
    end
    
    return 0. 
end

function I37integrand(lmbd,paralEdgePair::ParalEdgePair)
    return 0.
end

function I38(paralEdgePair::ParalEdgePair)
        
    function I38aux(s,ParalEdge3DPair)
        ans = 0.
        return ans; 
    end
    
    return 0. 
end

function I38integrand(lmbd,paralEdgePair::ParalEdgePair)
    return 0.
end

function I47(paralEdgePair::ParalEdgePair)
    return 0. 
end

function I47integrand(lmbd,paralEdgePair::ParalEdgePair)
    return 0.
end

function Qphi1theta1(paralEdgePair::ParalEdgePair)
    edgePair = paralEdgePair.edgePair
    e1       = edgePair.e1 
    e2       = edgePair.e2
    ans = I36(paralEdgePair)
    ans += I37(paralEdgePair)
    ans += I38(paralEdgePair)
    return ans 
end

function Qphi1theta1integrand(lmbd,paralEdgePair::ParalEdgePair)
    edgePair = paralEdgePair.edgePair
    e1 = edgePair.e1 
    e2 = edgePair.e2    
    r  = e1.r1+lmbd[1]*e1.tau 
    rp = e2.r1+lmbd[2]*e2.tau
    return (e1(r)[1])*(e2(rp)[1])/norm(r-rp)
end;

### Section 4.3: Case of Coinciding Lines or as Extension of Each Other 
* Analytically: currently fails as I3() returns Inf for point belonging to line;
* Numerically: currently fails as integrand singular near the lower bound; 

In [ ]:
function I27(coinEdgePair::CoinEdgePair)
    edgePair = coinEdgePair.edgePair
    e1       = edgePair.e1 
    e2       = edgePair.e2
    t1 = e1.len/2*I3(e1.r1,e2)
    t2 = dot(e2.r2-e1.r2,e2.tau)/2*Rphi1(e2.r2,e1)
    t3 = dot(e2.r1-e1.r2,e2.tau)/2*Rphi1(e2.r1,e1)
    return -t1-t2+t3 
end

function I27integrand(lmbd,coinEdgePair::CoinEdgePair)
    edgePair = coinEdgePair.edgePair
    e1       = edgePair.e1 
    e2       = edgePair.e2    
    r  = e1.r1+lmbd[1]*e1.tau 
    rp = e2.r1+lmbd[2]*e2.tau
    return dot(r,e1.e)/norm(r-rp)
end

function I3(a::Point3D,el::Edge3D)
    ##cutoff = 1e-18
    num  = norm(el.r2-a) + norm(el.r1-a) + norm(el.r2-el.r1)
    den  = norm(el.r2-a) + norm(el.r1-a) - norm(el.r2-el.r1)
    ##if (den<cutoff) den = cutoff end 
    return log(num/den)  
end 

function I41(coinEdgePair::CoinEdgePair)
    edgePair = coinEdgePair.edgePair
    e1       = edgePair.e1 
    e2       = edgePair.e2
    ans  = -dot(e1.r1-e1.r2,e2.tau)*(e2(e1.r2)[1]*I3(e1.r1,e2)-Rphi1(e1.r1,e2))
    ans += -dot(e2.r2-e1.r2,e2.tau)*Rphi1(e2.r2,e1)
    ans += dot(e2.r1-e1.r2,e2.tau)*Rphi1(e2.r1,e1)    
    return ans  
end

function I41integrand(lmbd,coinEdgePair::CoinEdgePair)
    edgePair = coinEdgePair.edgePair
    e1       = edgePair.e1 
    e2       = edgePair.e2 
    r  = e1.r1+lmbd[1]*e1.tau 
    rp = e2.r1+lmbd[2]*e2.tau
    return dot(r,e1.e)*dot(rp,e1.e)/norm(r-rp)
end

function Qphi1theta1(coinEdgePair::CoinEdgePair)
    edgePair = coinEdgePair.edgePair
    e1       = edgePair.e1 
    e2       = edgePair.e2
    return ans 
end

function Qphi1theta1integrand(lmbd,coinEdgePair::CoinEdgePair)
    edgePair = coinEdgePair.edgePair
    e1 = edgePair.e1 
    e2 = edgePair.e2    
    r  = e1.r1+lmbd[1]*e1.tau 
    rp = e2.r1+lmbd[2]*e2.tau
    return (e1(r)[1])*(e2(rp)[1])/norm(r-rp)
end;

In [ ]:
I41(coinEdgePair1)

In [ ]:
coinEdgePair1.edgePair.e1

In [ ]:
coinEdgePair1.edgePair.e2

In [ ]:
I27(coinEdgePair1)

In [ ]:
len1 = coinEdgePair1.edgePair.e1.len; len2 = coinEdgePair1.edgePair.e2.len
I27valnum = hcubature(lmbd -> I27integrand(lmbd,coinEdgePair1), (0,0), (len1,len2);rtol=1e-10,atol=1e-10)[1]  


In [ ]:
I41(coinEdgePair1)

In [ ]:
Itestintegrand([0.1,0.])

In [ ]:
I41integrand([0.1,0.],coinEdgePair1)

In [ ]:
arg  = coinEdgePair1
len1 = arg.edgePair.e1.len; len2 = arg.edgePair.e2.len
I41valnum = hcubature(lmbd -> Itestintegrand(lmbd), (0,0), (len1,len2);rtol=1e-10,atol=1e-10)[1]  

In [ ]:
arg  = coinEdgePair1
len1 = arg.edgePair.e1.len; len2 = arg.edgePair.e2.len
I41valnum = hcubature(lmbd -> I41integrand(lmbd,arg), (0,0), (len1,len2);rtol=1e-10,atol=1e-10)[1]  

In [ ]:
[len1, len2]

In [ ]:
Qphi1theta1(coinEdgePair1)